In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR = os.environ.get("ITDA_INPUT_DIR", "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
# Add src to Python module path
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from ocr_lab.contracts import OCRToken
from ocr_lab.modules.date_normalizer import DateNormalizer
from ocr_lab.modules.amsc_cascade_ocr import AMSC_CascadeOCRBackend
from ocr_lab.modules.union_spatial_selector import UnionSpatialSelector
# AMSC-OCR Championship Backend: Full PP-OCR + YOLO Expiry Crop True Union
weights_root = Path(os.environ.get("ITDA_WEIGHTS_ROOT", str(ROOT / "weights")))
det_dir = weights_root / "paddle" / "ppocrv5_mobile_det"
rec_dir = Path(os.environ.get("ITDA_V6_WEIGHTS_ROOT", str(weights_root / "paddle" / "PP-OCRv6_medium_rec")))
yolo_weight = Path(os.environ.get("ITDA_YOLO_WEIGHTS", str(weights_root / "final_kaggle" / "expiry_binary_yolov8n_1280_best.pt")))
if not yolo_weight.is_file():
    raise FileNotFoundError(f"Missing required YOLO weights: {yolo_weight}. Run bash download_weights.sh first.")
ocr_backend = AMSC_CascadeOCRBackend(
    runtime_device="cpu",
    fast_exit_enabled=True,
    fast_exit_conf=0.85,
    calendar_min_year=2020,
    calendar_max_year=2035,
    yolo_weights=str(yolo_weight),
    yolo_imgsz=960,
    yolo_conf=0.20,
    yolo_expand=1.0,
    yolo_margin=6,
    yolo_batch_size=8,
    yolo_max_boxes=4,
    yolo_rec_model_name="PP-OCRv6_medium_rec",
    yolo_rec_model_dir=str(rec_dir) if rec_dir.is_dir() else None,
    full_ocr_params={
        "require_local_weights": False,
        "text_detection_model_name": "PP-OCRv5_mobile_det",
        "text_recognition_model_name": "PP-OCRv6_medium_rec",
        "text_detection_model_dir": str(det_dir) if det_dir.is_dir() else None,
        "text_recognition_model_dir": str(rec_dir) if rec_dir.is_dir() else None,
        "det_max_side": 960,
        "det_thresh": 0.25,
        "box_thresh": 0.50,
        "unclip_ratio": 1.8,
        "recognition_batch_size": 1,
        "fallback_enabled": False,
        "engine": "paddle",
        "enable_mkldnn": False,
    },
)
print("Successfully initialized AMSC_CascadeOCRBackend.")
selector = UnionSpatialSelector(
    allow_day_first=True,
    allow_month_names=True,
    keyword_weight=0.7,
    negative_keyword_weight=0.45,
    confidence_weight=0.2,
    detection_confidence_weight=0.2,
    calendar_weight=0.5,
    pattern_weight=0.25,
    position_weight=0.25,
    partial_date_penalty=0.45,
    time_like_penalty=0.60,
    future_date_bonus=0.15,
    yolo_bonus=1.5,
    candidate_score_weight=1.5,
    recognition_log_weight=1.5,
    detection_log_weight=0.5,
    bbox_iou_weight=0.5,
)
normalizer = DateNormalizer(none_token="NONE", year_missing_token="NONE", day_missing_token="NONE")


In [ ]:
input_path = Path(INPUT_DIR)
valid_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
if input_path.is_dir():
    image_files = sorted([
        p for p in input_path.iterdir() 
        if p.is_file() and p.suffix.lower() in valid_extensions and not p.name.startswith("._")
    ])
else:
    image_files = []
print(f"Discovered {len(image_files)} images in {INPUT_DIR}")
results = []
for img_file in image_files:
    image_id = img_file.stem
    try:
        tokens = ocr_backend.extract(Path(img_file))
    except Exception as err:
        print(f"OCR inference error on {img_file.name}: {err}")
        tokens = []
    
    selected = selector.select(tokens)
    pred = normalizer.normalize(image_id, selected)
    results.append({
        "image_id": str(image_id),
        "year": str(pred.year),
        "month": str(pred.month),
        "day": str(pred.day),
        "final_date": str(pred.final_date),
    })
df = pd.DataFrame(results, columns=["image_id", "year", "month", "day", "final_date"])


In [ ]:
output_file = Path(OUTPUT_PATH)
output_file.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Successfully saved {len(df)} predictions to {OUTPUT_PATH}")
